# Sanity check: full fine-tune Whisper Medium (Mongolian)





## 1. Paths and imports

In [2]:
from datasets import load_dataset

ds = load_dataset("Ganaa0614/mongolian-commonvoice-stt-translated-full")
print(ds["train"].column_names)
print(ds["train"].features)

first_row = ds["train"][0]
print(first_row)

data/test-00000-of-00001.parquet:   0%|          | 0.00/108M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/107M [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/425M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/425M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/3642 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3642 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/29133 [00:00<?, ? examples/s]

['path', 'sentence', 'audio', 'sentence_en']
{'path': Value('large_string'), 'sentence': Value('large_string'), 'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'sentence_en': Value('string')}


ImportError: To support decoding audio data, please install 'torchcodec'.

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(".")
if not os.path.isfile(os.path.join(PROJECT_ROOT, "run_train.py")):
    PROJECT_ROOT = os.path.abspath("whisper-md-mn")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
assert os.path.isdir(os.path.join(PROJECT_ROOT, "src"))

PROJECT_ROOT: /home/toru2/Amara/Deep_learning/lab3/whisper-md-mn


In [2]:
import logging
import torch

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

print("PyTorch:", torch.__version__)
print("CUDA: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:    ", torch.cuda.get_device_name(0))
    print("BF16:   ", torch.cuda.is_bf16_supported())

PyTorch: 2.10.0+cu128
CUDA:  True
GPU:     NVIDIA GeForce RTX 5080
BF16:    True


## 2. Resolve `common_voice_mn`

In [3]:
from pathlib import Path

DATA_ROOT = None
for rel in ("../common_voice_mn", "common_voice_mn", "../../lab3/common_voice_mn"):
    cand = Path(PROJECT_ROOT) / rel
    if (cand / "validated.tsv").is_file() and (cand / "clips").is_dir():
        DATA_ROOT = cand.resolve()
        break
if DATA_ROOT is None:
    raise FileNotFoundError("Point DATA_ROOT to common_voice_mn")
print("DATA_ROOT:", DATA_ROOT)

DATA_ROOT: /home/toru2/Amara/Deep_learning/lab3/common_voice_mn


## 3. Load TSV and split (same as `src.data`)

In [ ]:
from IPython.display import display
import pandas as pd

from src.data import load_validated_frame, log_missing_clips, train_val_test_split_df

df = load_validated_frame(DATA_ROOT)
display(df.head(3))
log_missing_clips(df, DATA_ROOT / "clips")
df_train, df_val, df_test = train_val_test_split_df(df)

## 4. Why memmap?

Precomputing all log-mel features avoids holding huge Arrow tables in RAM. Each row is written as float16 to a `.bin` file; training reads slices via `numpy.memmap` (see `src/memmap.py`).

## 5. Processor + optional tiny cache (2 rows)

Building the full cache for all training rows takes a long time; this cell only checks that **preprocess** works on a **tiny** slice. For real training use `run_train.py` (or uncomment and run on full splits only when ready).

In [ ]:
from transformers import WhisperProcessor

from src.config import MODEL_ID
from src.memmap import preprocess_to_memmap

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="mongolian", task="transcribe")

cache_demo = Path(PROJECT_ROOT) / "sanity-cache-demo"
cache_demo.mkdir(exist_ok=True)

tiny_train = df_train.head(2).copy()
ds_demo = preprocess_to_memmap(
    tiny_train,
    "demo",
    DATA_ROOT / "clips",
    cache_demo,
    processor,
    processor.feature_extractor.sampling_rate,
)
print("demo samples:", len(ds_demo))
print("feature shape:", ds_demo[0]["input_features"].shape)

## 6. Collator batch shape

In [ ]:
from src.collate import DataCollatorSpeechSeq2SeqWithPadding

collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.bos_token_id,
)
batch = collator([ds_demo[0], ds_demo[1]])
print("input_features:", batch["input_features"].shape, batch["input_features"].dtype)
print("labels:        ", batch["labels"].shape)

## 7. Model parameters and weight size (full fine-tune)

`whisper-md-mn` trains **all** weights of `openai/whisper-medium` (no LoRA). Below: total / trainable parameter counts and approximate **static** weight size if stored in fp32 vs bf16 (training uses extra memory for optimizer states and activations).

In [ ]:
from transformers import WhisperForConditionalGeneration

from src.config import MODEL_ID

# Loads full weights once (uses HF cache after first download).
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.eval()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_layers_enc = model.config.encoder_layers
n_layers_dec = model.config.decoder_layers
d_model = model.config.d_model
vocab = model.config.vocab_size

print("checkpoint id:     ", MODEL_ID)
print("architecture:      ", model.config.model_type)
print("encoder / decoder: ", n_layers_enc, "/", n_layers_dec, "layers")
print("d_model:           ", d_model)
print("vocab_size:        ", vocab)
print("total parameters:  ", f"{total:,}  (~{total/1e6:.1f}M)")
print("trainable params:  ", f"{trainable:,}  (full FT → all trainable)")
assert total == trainable, "unexpected frozen params"

for name, bytes_per in ("fp32", 4), ("bf16/fp16", 2):
    approx_gb = total * bytes_per /1e9
    print(f"weights only ~{name}: {approx_gb:.2f} GB (order-of-magnitude)")

del model
import gc
gc.collect()

## 8. After training: what’s in `results/`

Training writes the **processor** at the start, **checkpoints** every `SAVE_STEPS`, TensorBoard under `results/runs/`, and the **feature cache** under `results/feature_cache/`. The **largest** artifacts are usually checkpoint folders (`pytorch_model*.bin` or `model.safetensors`).

In [ ]:
from pathlib import Path

def dir_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total


def human(n: int) -> str:
    x = float(max(n, 0))
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if x < 1024.0 or unit == "TB":
            return f"{int(x)} B" if unit == "B" else f"{x:.2f} {unit}"
        x /= 1024.0


RESULTS = Path(PROJECT_ROOT) / "results"
print("RESULTS:", RESULTS.resolve())

if not RESULTS.is_dir():
    print("(no results/ yet — run: python run_train.py)")
else:
    parts = []
    for child in sorted(RESULTS.iterdir()):
        sz = dir_size_bytes(child) if child.is_dir() else child.stat().st_size
        parts.append((sz, child.name))
    parts.sort(reverse=True)
    print(f"{'size':>12}  path")
    for sz, name in parts:
        print(f"{human(sz):>12}  {name}")
    print("TOTAL: ", human(dir_size_bytes(RESULTS)))

## Next steps

- **Sections 7–8** above: parameter count for Whisper Medium and disk usage under `results/`.
- Full preprocessing + training: `python run_train.py` (needs GPU, ~16GB+ free disk for feature cache).
- TensorBoard: `tensorboard --logdir results/runs`.
- After training: `python gradio_demo.py --model-path results` (or `--model-path results/checkpoint-XXXX` for a specific checkpoint).
- Remove `sanity-cache-demo/` if you do not need the tiny demo cache.